# Урок 2.3. Chroma: первая векторная база данных

**Модуль 2 «Embeddings и векторные базы данных» · Курс «LLM для Python-разработчиков»**

Теорию индексов вы прошли на собственном опыте — пора пользоваться готовым. Начинаем с **Chroma** — «SQLite мира векторов»: `pip install`, ноль инфраструктуры, файл на диске. Хитрость урока: механику базы мы изучим на **прозрачных четырёхмерных векторах, сделанных руками** — чтобы видеть насквозь, что происходит. А потом подключим настоящую модель и проиндексируем базу знаний «Векторики».

## 🎯 Цели урока

После этого урока вы сможете:

- создавать коллекции Chroma **с правильной метрикой** (спойлер: дефолт — не косинус);
- выполнять полный CRUD: `add`/`upsert`/`get`/`update`/`delete` — и знать, почему в ноутбуках живут на `upsert`;
- фильтровать поиск по **метаданным** (`where`) и тексту (`where_document`);
- включать **персистентность** одним аргументом и понимать её пределы;
- индексировать реальные документы с эмбеддингами e5 и **сверять качество eval'ом**;
- отличать задачи для Chroma от задач для Qdrant.

## План

1. Устройство Chroma: клиенты, коллекции, записи
2. Мини-коллекция на прозрачных векторах
3. Поиск и анатомия ответа
4. Метаданные и фильтры
5. CRUD и персистентность
6. Настоящая семантика: «Векторика» + e5 + eval

⏱ **Время:** ~60–75 минут.

> 💡 Вся механика БД в этом уроке работает сразу после `pip install chromadb`. Модель e5 нужна только для секции 6 — как обычно, эти ячейки вежливо пропустятся, если модель ещё не скачана (урок 2.1).

## 1. Устройство Chroma

Три вещи, которые нужно знать перед первым запросом.

**Клиенты** — три способа подключиться, прямая аналогия с SQLite:

| клиент | аналогия | когда |
|---|---|---|
| `EphemeralClient()` | `:memory:` | эксперименты, тесты |
| `PersistentClient(path=...)` | файл `.db` | ноутбуки, прототипы, мелкий прод |
| `HttpClient(...)` | клиент к серверу | Chroma как отдельный сервис |

**Коллекция** — аналог таблицы. Создаётся с настройками HNSW-индекса (урок 2.2 — теперь вы знаете, что там внутри).

**Запись** — четыре поля:

```text
id         "otpusk"                     — ваш ключ (строка), как PRIMARY KEY
embedding  [0.12, -0.53, ...]           — вектор
document   "Каждому сотруднику..."      — исходный текст (опционально)
metadata   {"topic": "hr", "year": 2026} — плоский словарь для фильтров
```

И одно принципиальное решение на весь курс: **эмбеддинги мы всегда считаем сами и передаём явно.** У Chroma есть «удобная» встроенная функция эмбеддингов, но она (а) скачает модель без спроса, (б) эта модель англоязычная — на русских текстах даст мусор без единой ошибки в логах, (в) вы теряете контроль над правилом «одна коллекция — одна модель» из урока 2.1. Explicit is better than implicit.

In [ ]:
# Единственная зависимость урока
%pip install -q chromadb

## 2. Мини-коллекция на прозрачных векторах

Настоящие эмбеддинги — 384 непрозрачных числа. Для изучения механики БД сделаем **интерпретируемые** векторы: всего 4 измерения, и мы сами назначим им смысл:

```text
ось 0: животные    ось 1: техника    ось 2: еда    ось 3: спорт
```

Каждый текст опишем степенью выраженности каждой «темы». Это тот же принцип, что у настоящих эмбеддингов, — просто у них оси выучены моделью и не имеют человеческих имён.

In [ ]:
import chromadb

client = chromadb.EphemeralClient()   # in-memory: идеально для экспериментов

# ВАЖНО: метрика по умолчанию - L2 (евклид). Нам нужен косинус - задаём явно!
mini = client.get_or_create_collection(
    name="mini",
    metadata={"hnsw:space": "cosine"},
)

#        id        текст                                      метаданные                    вектор [жив, тех, еда, спорт]
MINI_DOCS = [
    ("cat",    "Кот дремлет на подоконнике",              {"topic": "животные", "words": 4}, [0.95, 0.02, 0.01, 0.02]),
    ("dog",    "Пёс с азартом гоняет мяч во дворе",       {"topic": "животные", "words": 7}, [0.90, 0.02, 0.02, 0.35]),
    ("server", "Сервер перегрелся под нагрузкой",         {"topic": "техника",  "words": 4}, [0.02, 0.95, 0.01, 0.02]),
    ("db",     "База данных ждёт переиндексации",         {"topic": "техника",  "words": 4}, [0.05, 0.90, 0.02, 0.01]),
    ("pizza",  "Пицца с грибами уже остыла",              {"topic": "еда",      "words": 5}, [0.02, 0.02, 0.95, 0.05]),
    ("match",  "Финальный матч закончился со счётом 3:2", {"topic": "спорт",    "words": 6}, [0.10, 0.02, 0.05, 0.90]),
]

mini.upsert(   # upsert, а не add: повторный запуск ячейки не упадёт на дублях id
    ids=[d[0] for d in MINI_DOCS],
    documents=[d[1] for d in MINI_DOCS],
    metadatas=[d[2] for d in MINI_DOCS],
    embeddings=[d[3] for d in MINI_DOCS],
)
print(f"В коллекции {mini.count()} записей")

Два решения в этой ячейке стоят комментария:

- **`metadata={"hnsw:space": "cosine"}`** — без этого Chroma построит индекс по евклидову расстоянию. Для нормированных векторов ранжирование получится похожим, но пороги и числа «поедут», а для ненормированных сломается всё. Первая строчка в чек-листе отладки любого «странного поиска» в Chroma.
- **`upsert` вместо `add`** — `add` бросит ошибку на существующем id. Для идемпотентных ячеек ноутбука (и большинства пайплайнов индексации) `upsert` — правильный глагол.

## 3. Поиск и анатомия ответа

Запрос — тоже вектор. Составим его руками: «что-то про животных и немного про спорт»:

In [ ]:
result = mini.query(
    query_embeddings=[[0.80, 0.0, 0.0, 0.40]],   # животные + спорт
    n_results=3,
)

print("Сырой ответ (структура):", ", ".join(sorted(k for k in result if result[k])))
print()
for doc_id, doc, dist in zip(result["ids"][0],
                             result["documents"][0],
                             result["distances"][0]):
    print(f"  distance={dist:.3f}  similarity={1 - dist:.3f}  [{doc_id}] {doc}")

Разбор:

- Все поля ответа — **списки списков**: внешний уровень — по запросам (их можно передать пачкой), внутренний — по результатам. Отсюда вездесущее `result["ids"][0]`.
- Chroma возвращает **`distances` — расстояния, не близости**: cosine distance = 1 − cosine similarity, **меньше = ближе**. Помните ошибку №1 из урока 2.2? Вот она в дикой природе. Отсортируете по убыванию — получите худшие результаты с уверенным видом.
- Почему победил `dog`, а не «более животный» `cat`? У пса в векторе есть спортивная компонента (0.35 — он же гоняет мяч), а у запроса — спортивная ось 0.4. Косинус смотрит на **весь профиль** вектора. С настоящими эмбеддингами происходит ровно то же, просто по 384 осям сразу.

## 4. Метаданные и фильтры

Метаданные — плоский словарь (строки, числа, булевы) при каждой записи. Фильтр `where` отсекает кандидатов **по ходу поиска** — это не post-filter из урока 2.2, пустой выдачи из-за «чужих» результатов не будет (но если условию соответствует полтора документа — больше полутора и не вернётся).

Операторы: `$eq` (по умолчанию), `$ne`, `$gt/$gte/$lt/$lte`, `$in/$nin`, логические `$and/$or`. Плюс `where_document={"$contains": "..."}` — фильтр по подстроке в самом тексте:

In [ ]:
def show(label: str, result: dict) -> None:
    ids = result["ids"][0]
    print(f"{label:<46} -> {ids}")

query = [[0.80, 0.0, 0.0, 0.40]]   # тот же запрос "животные + спорт"

show("без фильтра",
     mini.query(query_embeddings=query, n_results=3))

show('where={"topic": "животные"}',
     mini.query(query_embeddings=query, n_results=3,
                where={"topic": "животные"}))

show('where={"words": {"$gte": 6}}',
     mini.query(query_embeddings=query, n_results=3,
                where={"words": {"$gte": 6}}))

show('where={"topic": {"$in": ["еда", "спорт"]}}',
     mini.query(query_embeddings=query, n_results=3,
                where={"topic": {"$in": ["еда", "спорт"]}}))

show('where_document={"$contains": "мяч"}',
     mini.query(query_embeddings=query, n_results=3,
                where_document={"$contains": "мяч"}))

Знакомая по SQL логика — только `WHERE` работает в связке с векторным поиском: сначала смысловая близость задаёт кандидатов, фильтр гарантирует соответствие условиям. В проде это главный инструмент мультиарендности («только документы этого клиента»), приватности («только доступные этому пользователю») и свежести («year >= 2026»).

## 5. CRUD и персистентность

Оставшиеся глаголы — как у любой БД:

In [ ]:
# READ по ключу - без всякого поиска
record = mini.get(ids=["pizza"], include=["documents", "metadatas"])
print("get('pizza'):", record["documents"], record["metadatas"])

# UPDATE через upsert: пицца "переехала" из еды на стадион - меняем и вектор
mini.upsert(
    ids=["pizza"],
    documents=["Пиццу доели на трибуне стадиона"],
    metadatas=[{"topic": "еда", "words": 5}],
    embeddings=[[0.02, 0.02, 0.60, 0.55]],
)
result = mini.query(query_embeddings=[[0.0, 0.0, 0.0, 1.0]], n_results=2)  # чистый спорт
print("топ-2 про спорт после upsert:", result["ids"][0])

# DELETE
mini.delete(ids=["match"])
print("после delete('match'):", mini.count(), "записей")

Обратите внимание: после `upsert` пицца начала находиться по «спортивным» запросам — **обновление записи меняет результаты поиска немедленно**, HNSW-индекс (урок 2.2) правится инкрементально, без пересборки.

Теперь персистентность. Меняется ровно одна строка — и это до степени смешения похоже на переход от `sqlite3.connect(":memory:")` к файлу:

In [ ]:
import chromadb

DB_PATH = "chroma_demo"   # папка появится рядом с ноутбуком; удалить = удалить базу

pc = chromadb.PersistentClient(path=DB_PATH)
knowledge = pc.get_or_create_collection("knowledge", metadata={"hnsw:space": "cosine"})
knowledge.upsert(
    ids=["policy"],
    documents=["Отпуск согласуется за 14 дней"],
    metadatas=[{"topic": "hr", "version": 1}],
    embeddings=[[0.9, 0.1, 0.0, 0.0]],
)

# имитируем перезапуск приложения: новый клиент к той же папке
del pc, knowledge
pc2 = chromadb.PersistentClient(path=DB_PATH)
survived = pc2.get_collection("knowledge")
print(f"после 'перезапуска': {survived.count()} записей —",
      survived.get(ids=["policy"])["documents"][0])

Никаких `commit`/`save` — Chroma пишет на диск сама. И ровно как у SQLite, у файловой природы есть предел: **один процесс-писатель**, никакой репликации, никакого «подключиться с другого сервера» (для этого есть server-режим, но там, где он нужен, обычно уже пора в Qdrant — урок 2.4).

## 6. Настоящая семантика: «Векторика» + e5 + eval

Механику освоили на игрушечных осях — теперь всё то же самое, но с настоящей моделью из урока 2.1 и 14 документами «Векторики». Схема любого пайплайна индексации:

```text
документы -> эмбеддинги (passage:) -> upsert в коллекцию (id, вектор, текст, метаданные)
вопрос    -> эмбеддинг  (query:)   -> query + where-фильтры -> топ-k
```

In [ ]:
import numpy as np

EMB_MODEL_NAME = "intfloat/multilingual-e5-small"

model = None
try:
    from sentence_transformers import SentenceTransformer
    from huggingface_hub import snapshot_download
    snapshot_download(EMB_MODEL_NAME, local_files_only=True)  # только из кэша
    model = SentenceTransformer(EMB_MODEL_NAME)
    print("Модель загружена — секция будет живой.")
except Exception:
    print("Модель e5 не найдена в кэше — установка и скачивание описаны в уроке 2.1.")
    print("Ячейки секции 6 пропустятся, механика БД от этого не страдает.")

EMB_AVAILABLE = model is not None

def embed_texts(texts: list[str], *, prefix: str) -> np.ndarray:
    """Эмбеддинги с обязательным для e5 префиксом (query:/passage:), урок 2.1."""
    return model.encode([f"{prefix}: {t}" for t in texts], normalize_embeddings=True)

In [ ]:
from utils import load_documents

docs = load_documents("data")
print(f"Загружено {len(docs)} документов «Векторики»")

if EMB_AVAILABLE:
    vectorika = client.get_or_create_collection(
        "vectorika", metadata={"hnsw:space": "cosine"})
    vectors = embed_texts([d.text for d in docs], prefix="passage")
    vectorika.upsert(
        ids=[d.doc_id for d in docs],
        documents=[d.text for d in docs],
        metadatas=[{"topic": d.topic, "title": d.title, "source": d.source}
                   for d in docs],
        embeddings=vectors.tolist(),
    )
    print(f"Проиндексировано: {vectorika.count()} документов")

    def search(question: str, n: int = 3, where: dict | None = None) -> None:
        q_vec = embed_texts([question], prefix="query")[0]
        res = vectorika.query(query_embeddings=[q_vec.tolist()],
                              n_results=n, where=where)
        print(f"\nВопрос: {question}" + (f"   [фильтр: {where}]" if where else ""))
        for doc_id, meta, dist in zip(res["ids"][0], res["metadatas"][0],
                                      res["distances"][0]):
            print(f"  sim={1 - dist:.3f}  {doc_id:<14} {meta['title']}")

    search("Как оформить отгул на пару дней?")
    search("Сломался рабочий ноутбук, куда бежать?")
    search("Сломался рабочий ноутбук, куда бежать?", where={"topic": "it"})
else:
    print("[семантическая часть пропущена: модель не загружена — урок 2.1]")

Всё как в уроке 2.1 — только вместо самодельного перебора с numpy теперь коллекция с HNSW-индексом, метаданными, фильтрами и персистентностью за одну строку. Осталось убедиться, что при переезде мы ничего не потеряли, — прогоним **тот же eval из урока 2.1**:

In [ ]:
EVAL_QUESTIONS = [
    ("Сколько дней отпуска положено в году?",              "otpusk"),
    ("Как оформить больничный лист?",                      "bolnichny"),
    ("VPN подключён, но внутренние сайты не открываются",  "vpn"),
    ("Кто должен посмотреть мой код перед мержем?",        "code-review"),
    ("Упал прод, что делать в первую очередь?",            "incidents"),
]

if EMB_AVAILABLE:
    top1 = top3 = 0
    for question, expected in EVAL_QUESTIONS:
        q_vec = embed_texts([question], prefix="query")[0]
        res = vectorika.query(query_embeddings=[q_vec.tolist()], n_results=3)
        top_ids = res["ids"][0]
        top1 += top_ids[0] == expected
        top3 += expected in top_ids
    total = len(EVAL_QUESTIONS)
    print(f"top-1 accuracy: {top1}/{total}   top-3 hit rate: {top3}/{total}")
    print("Числа должны совпасть с уроком 2.1: модель та же, метрика та же,")
    print("а на 14 документах HNSW работает как точный поиск.")
else:
    print("[пропущено: модель не загружена — урок 2.1]")

## Когда Chroma заканчивается

Chroma закрывает прототипы, ноутбуки, внутренние инструменты и мелкий прод до ~миллиона векторов. Признаки, что пора на Qdrant (следующий урок):

- к базе ходят **несколько сервисов** по сети — нужен настоящий сервер с API и авторизацией;
- **миллионы векторов**, шардирование, репликация, снапшоты;
- тяжёлые **фильтрованные запросы на масштабе** (помните filtered HNSW из урока 2.2);
- нужны **квантизация** для экономии RAM, мониторинг, метрики.

Знакомая развилка? Это SQLite → PostgreSQL, шаг в шаг, — и docker-compose для этого шага уже лежит в папке модуля.

## ⚠️ Частые ошибки

1. **Забыть `hnsw:space: "cosine"`.** Дефолт — L2. Числа в `distances` станут другими, пороги отсечения поедут, а на ненормированных векторах развалится и ранжирование.
2. **`query_texts` без своей embedding-функции.** Chroma молча скачает встроенную английскую модель и посчитает «эмбеддинги» русских текстов. Ошибок не будет — будет плохой поиск. Передавайте векторы сами.
3. **Путать `distances` и similarity.** Меньше = ближе. Отсортировали по убыванию — получили анти-поиск.
4. **`add` в идемпотентном коде.** Повторный запуск — `DuplicateIDError`. Пайплайны индексации живут на `upsert`.
5. **Смешать модели в коллекции** (урок 2.1). Проиндексировали e5, а запросы считаете другой моделью — тихий мусор. Модель — часть схемы коллекции; смените модель — пересоздайте коллекцию.
6. **Два процесса на одну папку `PersistentClient`.** Это файловая БД, как SQLite: один писатель. Нужен доступ по сети — server-режим или Qdrant.

## 🏋️ Упражнения

### Упражнение 1 — почувствуй оси 🎛

Добавьте в мини-коллекцию свою запись с 4-мерным вектором (например, «велосипед у серверной стойки» — техника + спорт) и подберите вектор запроса, который найдёт её **первой**, не сломав остальные результаты. Проверьте query.

### Упражнение 2 — составные фильтры 🔍

На мини-коллекции соберите запрос с `$and`, `$in` и `$gte` одновременно: «похожее на животных, из тем "животные" или "спорт", длиной от 6 слов». Затем инвертируйте: те же условия, но `$nin`. Убедитесь, что выдача объяснима.

### Упражнение 3 — версии документов 📝

В персистентной коллекции `knowledge` обновите политику отпусков до версии 2 (новый текст, `{"version": 2}`) через `upsert` и покажите, что (а) старого текста больше не найти, (б) метаданные отражают версию. Подумайте: а если нужно **хранить обе версии** и искать только по последней — как бы вы это устроили? (Подсказка: id и фильтры.)

## 📝 Мини-квиз

**1. Чем `PersistentClient` отличается от `EphemeralClient` и что у Chroma общего со SQLite?**

<details><summary>Ответ</summary>

`EphemeralClient` живёт в памяти процесса (аналог `:memory:`), `PersistentClient` пишет в папку на диске — как SQLite в файл: та же лёгкость, тот же предел (один процесс-писатель, нет сети/репликации).

</details>

**2. Вы забыли указать метрику при создании коллекции. Что произошло и чем грозит?**

<details><summary>Ответ</summary>

Индекс построился по умолчанию — на **L2** (евклид). Для нормированных векторов ранжирование почти не пострадает (L2² = 2 − 2·cos), но значения `distances` и все пороги станут другими; для ненормированных — сломается и порядок. Метрика — часть схемы, задаётся при создании.

</details>

**3. Запрос вернул `distances = [0.12, 0.48]`. Какая запись ближе и каковы similarity?**

<details><summary>Ответ</summary>

Ближе первая: это **расстояния**, меньше = ближе. Cosine similarity = 1 − distance: 0.88 и 0.52.

</details>

**4. Почему в этом курсе эмбеддинги передаются в Chroma явно, а не через встроенную функцию?**

<details><summary>Ответ</summary>

Контроль: встроенная функция тянет англоязычную модель (на русском — тихий мусор), скачивает её без спроса, и вы теряете гарантию «одна коллекция — одна модель» и e5-префиксы. Явные векторы = воспроизводимость и переносимость (тот же код переедет на Qdrant).

</details>

**5. Назовите три признака, что проекту пора с Chroma на Qdrant.**

<details><summary>Ответ</summary>

Несколько сервисов ходят к базе по сети; миллионы векторов и нужна репликация/шардирование/снапшоты; тяжёлые фильтрованные запросы или экономия RAM через квантизацию. Это переход SQLite → PostgreSQL.

</details>

## ✅ Решения упражнений

### Решение 1 — почувствуй оси

In [ ]:
mini.upsert(
    ids=["bike"],
    documents=["Велосипед прислонён к серверной стойке"],
    metadatas=[{"topic": "техника", "words": 5}],
    embeddings=[[0.03, 0.55, 0.02, 0.60]],   # техника + спорт, ни одна ось не доминирует
)

# запрос должен смотреть в ту же "диагональ" техника+спорт:
result = mini.query(query_embeddings=[[0.0, 0.6, 0.0, 0.6]], n_results=3)
for doc_id, dist in zip(result["ids"][0], result["distances"][0]):
    print(f"  sim={1 - dist:.3f}  {doc_id}")

Ключевая интуиция: запись находится первой, когда **профиль запроса повторяет профиль записи** — косинусу важны пропорции осей, а не абсолютные значения. Запрос «чистая техника» `[0, 1, 0, 0]` нашёл бы `server`, а не велосипед.

### Решение 2 — составные фильтры

In [ ]:
query = [[0.9, 0.0, 0.0, 0.2]]   # "в основном животные"

result = mini.query(
    query_embeddings=query, n_results=5,
    where={"$and": [
        {"topic": {"$in": ["животные", "спорт"]}},
        {"words": {"$gte": 6}},
    ]},
)
print("животные|спорт И words>=6 :", result["ids"][0])

result = mini.query(
    query_embeddings=query, n_results=5,
    where={"$and": [
        {"topic": {"$nin": ["животные", "спорт"]}},
        {"words": {"$gte": 5}},
    ]},
)
print("НЕ животные|спорт И words>=5:", result["ids"][0])

Первая выдача — только пёс: кот отсеян по длине (4 слова), а матч удалён ещё в секции CRUD. Вторая — велосипед из упражнения 1 и пицца: все записи вне «животных/спорта» с 5+ словами (сервер и база отсеяны по длине). Фильтры отработали *вместе* с векторным ранжированием — гибрид, ради которого метаданные и существуют.

### Решение 3 — версии документов

In [ ]:
knowledge = pc2.get_collection("knowledge")

# обновляем политику до версии 2
knowledge.upsert(
    ids=["policy"],
    documents=["Отпуск согласуется в HR-портале минимум за 7 дней"],
    metadatas=[{"topic": "hr", "version": 2}],
    embeddings=[[0.9, 0.15, 0.0, 0.0]],
)

record = knowledge.get(ids=["policy"], include=["documents", "metadatas"])
print("Текущий текст:  ", record["documents"][0])
print("Метаданные:     ", record["metadatas"][0])
print("Записей в базе: ", knowledge.count(), "- старой версии не существует")

`upsert` по тому же id — это **перезапись**: старого текста в базе больше нет, что для базы знаний обычно и нужно. А если требуется история версий — храните каждую версию отдельной записью (`policy_v1`, `policy_v2`) с метаданными `{"version": 1, "latest": False}` и ищите с фильтром `where={"latest": True}`. Знакомый по реляционным БД паттерн — soft versioning — работает и здесь.

> 🧹 Папку `chroma_demo` рядом с ноутбуком можно удалить в любой момент — это вся ваша база целиком, в этом и прелесть.

## 🎓 Итоги

- Chroma — SQLite мира векторов: `EphemeralClient` для экспериментов, `PersistentClient(path=...)` для персистентности без инфраструктуры.
- Коллекция создаётся **с метрикой** (`hnsw:space: "cosine"`), запись = id + вектор + документ + метаданные. Эмбеддинги передаём явно — контроль модели важнее удобства.
- `query` возвращает **расстояния** (меньше = ближе, similarity = 1 − distance), фильтры `where`/`where_document` работают в связке с поиском, а не после него.
- CRUD полный, `upsert` — рабочая лошадка идемпотентной индексации; HNSW обновляется инкрементально.
- Eval из урока 2.1 подтвердил: переезд на БД ничего не сломал. Механика проверена прозрачными векторами, семантика — настоящей моделью.

**Дальше — урок 2.4 «Переход на Qdrant»:** поднимем настоящий сервер векторного поиска в Docker (compose-файл уже в папке модуля), перенесём «Векторику» и посмотрим, что даёт продакшен-класс: payload-индексы, фильтрация внутри HNSW, снапшоты.

## 📚 Что почитать

- [Документация Chroma](https://docs.trychroma.com/) — коллекции, клиенты, деплой
- [Chroma Cookbook](https://cookbook.chromadb.dev/) — практичные рецепты и грабли
- [Метаданные и фильтры](https://docs.trychroma.com/docs/querying-collections/metadata-filtering) — полный список операторов
- [Тот самый HNSW внутри Chroma](https://docs.trychroma.com/docs/collections/configure) — параметры `hnsw:*`, которые вы теперь понимаете после урока 2.2